##  /data/.home


In [ ]:
%%bash
echo "home link make..."
d="/data/.home"
while IFS= read -r i; do
	rm -fr "$HOME/$(basename "$i")"
	ln -sf "$i" "$HOME/$(basename "$i")"
done < <(find "$d" -mindepth 1 -maxdepth 1 -type d)


## python & ipykernel

In [ ]:
%%bash
sudo pacman -S --needed --noconfirm python-pip python-ipykernel


## [vscode](../s/vscode.ipynb)


In [ ]:
%%bash
# download tar.gz
# https://code.visualstudio.com/Download
cd ~/Downloads
# tar xf *.tar.gz
rm *tar.gz
cd VSCode-linux-x64

# microsoft 3124568493@qq.com

### gpg

In [ ]:
%%bash
name="kefu"
email="kefu1820@gmail.com"
gd="${GPG_DIR:-$HOME/.gnupg}"

if gpg --list-keys "$email" &; then
	echo "✓ GPG key already exists for $email"
else
	read -sp "Enter GPG passphrase: " pswd
	echo
	
	chmod 700 "$gd"
	
	tmp=$(mktemp)
	cat >"$tmp" <<EOF
%echo Generating GPG key
Key-Type: RSA
Key-Length: 4096
Subkey-Type: RSA
Subkey-Length: 4096
Name-Real: $name
Name-Email: $email
Expire-Date: 3y
Passphrase: $pswd
%commit
%echo done
EOF
	
	gpg --batch --generate-key "$tmp"
	rm -f "$tmp"
	
	find "$gd" -type f -exec chmod 600 {} \;
	
	echo "✓ GPG key generated for $email"
	gpg --list-keys "$email"
fi


### firefox 

In [ ]:
# kefu51252@gmail.com

## chinese mirrors


In [ ]:
%%bash
sudo tee /etc/pacman.d/mirrorlist <<'EOF'
# China mirrors tsinghua
Server = https://mirrors.tuna.tsinghua.edu.cn/manjaro/stable/$repo/$arch
EOF


## yay


In [ ]:
%%bash
sudo pacman -Sy --needed --noconfirm base-devel yay 


In [ ]:
%%bash
dir="$HOME/.config/yay"
mkdir -p "$dir"
tee "$dir/config.json" > /dev/null << 'EOF'
{
    "editor": "nano",
    "pacmanbin": "pacman",
    "pacmanconf": "/etc/pacman.conf",
    "answerclean": "All",
    "removemake": "ask",
    "maxconcurrentdownloads": 5,
    "cleanAfter": false,
    "batchinstall": true,
    "DevelCheckUpdate": false
}
EOF
echo "yay config.json created successfully."


## pacman


In [ ]:
%%bash
PACMAN_FILE=/etc/pacman.conf
options=("Color" "ILoveCandy" "ParallelDownloads = 5")

for opt in "${options[@]}"; do
	key=${opt%% *}
	sudo sed -i "/^#\?$key/d; /^\[options\]/a $opt" "$PACMAN_FILE"
done

echo "✓ pacman options configured"


###  archlinuxcn


In [ ]:
%%bash
PACMAN_FILE=/etc/pacman.conf
Server="https://mirrors.tuna.tsinghua.edu.cn/archlinuxcn/\$arch"

if ! grep -q "^\[archlinuxcn\]" "$PACMAN_FILE"; then
	echo -e "\n[archlinuxcn]\nServer = $Server" | sudo tee -a "$PACMAN_FILE" 
else
	sudo sed -i "/^\[archlinuxcn\]/,/^\[/{/^Server/d}" "$PACMAN_FILE"
	sudo sed -i "/^\[archlinuxcn\]/a Server = $Server" "$PACMAN_FILE"
fi

echo "✓ archlinuxcn configured: $Server"


##  pacman-key


In [ ]:
%%bash
echo "installing pacman-key..."
sudo pacman -S --needed --noconfirm manjaro-keyring archlinux-keyring archlinuxcn-keyring 
sudo pacman-key --init 
sudo pacman-key --populate archlinux manjaro archlinuxcn 
sudo pacman -Syy --noconfirm 


## update system


In [ ]:
%%bash
sudo pacman -Syyu --noconfirm 


In [ ]:
%%bash
yay -Syyu --noconfirm 


In [ ]:
%%bash
yay -S --needed --noconfirm visual-studio-code-bin 


## reboot


## backup

In [ ]:
%%bash
snapshot_dir="/.snapshots"
sudo mkdir -p "$snapshot_dir"
snapshot_name="$snapshot_dir/update_system"
if [[ -d "$snapshot_name" ]]; then
    echo "  ✓ Snapshot already exists: $snapshot_name"
else
    sudo btrfs subvolume snapshot -r / "$snapshot_name"
fi

## google-chrome
## keepassxc
## cryptomator


In [ ]:
%%bash
yay -S --needed --noconfirm google-chrome 
# kefu1820@gmail.com

sudo pacman -S --needed --noconfirm keepassxc 

yay -S --needed --noconfirm cryptomator-bin 


## clash-verge-rev 

In [ ]:
%%bash
sudo pacman -S --needed --noconfirm clash-verge-rev 
# sub link : https://zhuzhuzhu.whtjdasha.com/api/v1/client/subscribe?token=eebe36f8c2eb695b9841a61eb4b03825
# setting : auto start; slient start ; allow lan;

In [ ]:
%%bash
cryptomator &
keepassxc &
code . &
clash-verge &
i=$(ip addr show | grep -E 'inet.*global' | awk '{print $2}' | cut -d'/' -f1 | head -n1) && echo "Using IP: $i "
google-chrome-stable --proxy-server="socks5://${i}:7897" &
echo "Done!!!"


## fcitx5


In [ ]:
%%bash
echo "installing fcitx5..."
sudo pacman -S --needed --noconfirm \
	fcitx5 \
	fcitx5-gtk \
	fcitx5-qt \
	fcitx5-configtool \
	fcitx5-chinese-addons \
	fcitx5-pinyin-zhwiki 
kwriteconfig6 --file kwinrc --group Wayland --key 'InputMethod' /usr/share/applications/org.fcitx.Fcitx5.desktop


## git & ssh


###  Git


In [ ]:
%%bash
name="kefu"
email="19157521820@163.com"
signingkey=$(gpg --list-secret-keys --keyid-format SHORT 2| grep sec | awk '{print $2}' | cut -d'/' -f2 | head -1)

git config --global user.name "$name"
git config --global user.email "$email"
git config --global init.defaultBranch "main"
git config --global gpg.program "gpg"
git config --global user.signingkey "$signingkey"
git config --global commit.gpgsign "false"
git config --global credential.helper "store"

echo "✓ Git configured"


###  SSH


In [ ]:
%%bash
email="19157521820@163.com"
ssh_key_type="ed25519"
ssh_dir="/data/.home/.ssh"

mkdir -p "$ssh_dir"
ssh_real_path=$(readlink -f "$ssh_dir")
ssh_key_path="$ssh_real_path/id_$ssh_key_type"

if [[ ! -f "$ssh_key_path" ]]; then
	ssh-keygen -t "$ssh_key_type" -C "$email" -f "$ssh_key_path" -N ""
	echo "✓ SSH key generated"
else
	echo "✓ SSH key already exists"
fi

chmod 700 "$ssh_real_path"
chmod 600 "$ssh_real_path"/id_* || true
chmod 644 "$ssh_real_path"/*.pub || true
[[ -f "$ssh_real_path/config" ]] && chmod 600 "$ssh_real_path/config"

ssh_home_path="$HOME/.ssh"
rm -rf "$ssh_home_path"
ln -sf "$ssh_real_path" "$ssh_home_path"


## 配置自动启动


In [ ]:
%%bash
autostart_dir="$HOME/.config/autostart"
mkdir -p "$autostart_dir"

app="cryptomator"
exec_path=$(which "$app" )

if [[ -n "$exec_path" ]]; then
	cat > "$autostart_dir/$app.desktop" <<EOF
[Desktop Entry]
Type=Application
Name=$app
Exec=$exec_path
Icon=$app
Comment=Auto-start $app
X-GNOME-Autostart-enabled=true
StartupNotify=false
Terminal=false
EOF
	echo "Created autostart file for $app"
else
	echo "Failed to find $app"
fi


## ip


In [3]:
%%bash
i=$(ip addr show | grep -E 'inet.*global' | awk '{print $2}' | cut -d'/' -f1 | head -n1) 
echo $i
el="export all_proxy=socks5://${i}:7897"
echo $el

192.168.0.103
export all_proxy=socks5://192.168.0.103:7897
